In [1]:
import pandas as pd

perf = pd.read_csv("../data/performance_data.csv")

REQ = {
    "kit_min": 8,                     # grease resistance
    "wvtr_max": 100,                  # g/m2/day
    "wvtr_conditions": "23 C, 50% RH",
    "contact_angle_min": 90,          # water repellence (hydrophobic)
    "coat_weight_max": 10,            # g/m2
}
ALLOW_PARTIAL_BIO = True

In [2]:
bio = ("yes", "partial") if ALLOW_PARTIAL_BIO else ("yes",)
cand = perf[perf["bio_based"].str.startswith(bio)].copy()
print("Candidates:", len(cand))

def check(value, op, limit):
    if pd.isna(value):
        return None
    return bool(value >= limit) if op == ">=" else bool(value <= limit)

def evaluate(row):
    wvtr_ok = None
    if str(row["wvtr_conditions"]).startswith(REQ["wvtr_conditions"]):
        wvtr_ok = check(row["wvtr"], "<=", REQ["wvtr_max"])
    return pd.Series({
        "kit_ok": check(row["kit"], ">=", REQ["kit_min"]),
        "wvtr_ok": wvtr_ok,
        "ca_ok": check(row["water_contact_angle_deg"], ">=", REQ["contact_angle_min"]),
        "cw_ok": check(row["coating_weight_gsm"], "<=", REQ["coat_weight_max"]),
    })

Candidates: 32


In [3]:
checks = cand.apply(evaluate, axis=1)
cand = pd.concat([cand, checks], axis=1)

OK = ["kit_ok", "wvtr_ok", "ca_ok", "cw_ok"]
cand["met"] = (cand[OK] == True).sum(axis=1)
cand["failed"] = (cand[OK] == False).sum(axis=1)
cand["unknown"] = cand[OK].isna().sum(axis=1)

ranked = cand.sort_values(["failed", "met", "coating_weight_gsm"],
                          ascending=[True, False, True])

show = ["paper_short", "sample_id", "base_material", "additive", "kit", "wvtr",
        "water_contact_angle_deg", "coating_weight_gsm", "met", "failed", "unknown"]
pd.set_option("display.max_colwidth", 30)
ranked[show].head(10)

,paper_short,sample_id,base_material,additive,kit,wvtr,water_contact_angle_deg,coating_weight_gsm,met,failed,unknown
31,Roman et al. 2023 (xylan d...,P9,acetylated xylan,chitosan 50%,8.0,30.0,92.80,5.0,4,0,0
18,Jahangiri et al. 2025 (nat...,Biowax,TopScreen biowax,NaN,NaN,519.1,105.28,8.2,2,0,2
19,Jahangiri et al. 2025 (nat...,Soywax,soy wax,NaN,NaN,683.5,108.86,9.7,2,0,2
17,Jahangiri et al. 2025 (nat...,Beeswax,beeswax,NaN,NaN,283.7,107.72,10.0,2,0,2
14,Yadav et al. 2024 (degrada...,P-W-LA4,polymeric wax (stearic aci...,on 5% starch precoat; solv...,12.0,NaN,111.65,NaN,2,0,2
15,Yadav et al. 2024 (degrada...,P-W-LA3-Neat,polymeric wax (stearic aci...,on 5% starch precoat; solv...,12.0,NaN,NaN,NaN,1,0,3
28,Roman et al. 2023 (xylan d...,P6,acetylated xylan,chitosan 25%,7.0,80.0,91.90,5.0,3,1,0
32,Roman et al. 2023 (xylan d...,P10,chitosan,NaN,6.0,20.0,90.20,5.0,3,1,0
9,Kumar et al. 2020 (PVA + MFC),PVA-MFC,polyvinyl alcohol (PVA),MFC 10% (90:10),9.0,NaN,46.00,2.5,2,1,1
4,Santos et al. 2023 (starch...,S.B7,cationic starch,bentonite 7 wt%; glycerol ...,12.0,1260.0,92.00,15.0,2,1,1


In [4]:
ranked.to_csv("../data/candidates_ranked.csv", index=False, encoding="utf-8")
print("Saved: data/candidates_ranked.csv")

Saved: data/candidates_ranked.csv
